<a href="https://colab.research.google.com/github/SummerSnow-flake/Linear_programming/blob/term_project/Credit_Solver_fin.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Credit Solver



In [ ]:
# Install and import libraries
!pip -q install pulp plotly pandas numpy matplotlib

import pulp
import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.io as pio
from itertools import combinations

pio.renderers.default = "colab"
pd.set_option("display.precision", 4)


In [ ]:
# Courses
courses = [
    "ANA3", "NMA1", "DIMA3", "LIP", "VOAF",
    "PRC1", "JAZ1", "JAZ2", "TEF1", "SSM1",
    "SOS1", "ZEL1", "TV1"
]

data = pd.DataFrame({
    "hours":        [8, 4, 2, 3, 6, 4, 2, 2, 4, 2, 2, 3, 2],
    "credits":       [9, 4, 2, 3, 6, 4, 2, 2, 4, 2, 2, 3, 1],
    "exams":       [1, 1, 1, 1, 1, 0, 0, 0, 1, 0, 0, 1, 0],
    "categories":     ["m","m","m","n","f","n","j","j","f","m","n","n","v"],
    "t-c ratio":           [3,2,2,2,2,2,2,2,2,2,2,2,1],
    "popularity":    [75,65,60,65,65,70,65,45,60,55,55,60,70],
    "coef fatigue":    [10,5,8,4,8,4,4,4,7,3,1,3,1],
    "required":     [1,1,1,1,1,1,1,1,0,0,0,0,0],
    "fatigue":         [24,14,15,12,19,8,6,6,16,5,3,11,2],
    "satisfaction without#":   [78,68,53,68,66,85,76,56,44,53,57,50,70],
}, index=courses)

display(data.style.format(precision=3).set_caption("Courses data"))

#Requirements
requirements = {
    #Max součet kreditů
    "max_credits": 38,
    #Min součet kreditů
    "min_credits": 30,
    #Max počet zkoušek
    "max_exams": 6,
    #Min počet zkoušek
    "min_exams": 3,
    #Max součet hodin
    "max_hours": 40,
    #Požadavky na kategorie
    "min_m": 2, #alespoň 2 předměty kategorie "m"
    "min_f": 1, #alespoň 1 předmět kategorie "f"
    "min_n": 1, #alespoň 1 předmět kategorie "n"
    "count_j": 2 #právě 2 předměty kategorie "j"
}
req1 = {
    "min_credits": 30,
    "min_exams": 3,
    "max_credits": 38,
    "max_exams": 6,
    "max_hours": 40
}
req_cat = {
    "min_m": 2,
    "min_f": 1,
    "min_n": 1,
    "min_j": 2,
    "max_j": 2
}
req_df = pd.DataFrame.from_dict(requirements, orient = "index", columns=["value"])
display(req_df.style.set_caption("Requirements"))


,hours,credits,exams,categories,t-c ratio,popularity,coef fatigue,required,fatigue,satisfaction without#
ANA3,8,9,1,m,3,75,10,1,24,78
NMA1,4,4,1,m,2,65,5,1,14,68
DIMA3,2,2,1,m,2,60,8,1,15,53
LIP,3,3,1,n,2,65,4,1,12,68
VOAF,6,6,1,f,2,65,8,1,19,66
PRC1,4,4,0,n,2,70,4,1,8,85
JAZ1,2,2,0,j,2,65,4,1,6,76
JAZ2,2,2,0,j,2,45,4,1,6,56
TEF1,4,4,1,f,2,60,7,0,16,44
SSM1,2,2,0,m,2,55,3,0,5,53


,value
max_credits,38
min_credits,30
max_exams,6
min_exams,3
max_hours,40
min_m,2
min_f,1
min_n,1
count_j,2


In [ ]:
def credit_solver(data: pd.DataFrame, requirements: dict, req1 : dict, req_cat : dict, w_choice = 'A'):
  problem = pulp.LpProblem("Credit_Solver", pulp.LpMaximize)
  course_choice = {course : pulp.LpVariable(course, cat=pulp.LpBinary) for course in data.index} #problem variables - subjects - binary choice {0 - not chosen, 1 chosen}
  ftg_max = pulp.LpVariable("ftg_max", lowBound = 0) #extra variable to replace max(0, credit_sum-30) in fatigue function
  # Satisfaction
    #Factors of satisfaction
  credit_sum = pulp.lpSum(data.loc[course, "credits"] * course_choice[course] for course in data.index)
  pop_sum = pulp.lpSum(data.loc[course, "popularity"] * course_choice[course] for course in data.index)
  hours_sum = pulp.lpSum(data.loc[course, "hours"] * course_choice[course] for course in data.index)
  #weights of factors in the objective function
  weights = {
    "A" : {"wc": 0.1, "wp": 0.02, "wh": 1.0, "wf": 1.0, "alpha": 2},
    "B": {"wc": 0.05, "wp": 0.05, "wh": 0.8, "wf": 1.2, "alpha": 2},
    "C": {"wc": 0.5,  "wp": 0.02, "wh": 0.8, "wf": 1.5, "alpha": 3},
    "D": {"wc": 0.0,  "wp": 0.02, "wh": 1.2, "wf": 1.0, "alpha": 4},
    "E": {"wc": 1.0,  "wp": 0.02, "wh": 0.5, "wf": 0.2, "alpha": 2},
    "F": {"wc": 0.2,  "wp": 0.1,  "wh": 0.8, "wf": 0.8, "alpha": 2},
    "G": {"wc": 0.05, "wp": 0.02, "wh": 1.0, "wf": 0.5, "alpha": 3},
    "H": {"wc": 0.3,  "wp": 0.03, "wh": 0.9, "wf": 0.7, "alpha": 2.5}
             }
  W = weights[w_choice]

  #replacement of max in fatigue function
  problem += ftg_max >= credit_sum -30
  problem += ftg_max >= 0
  fatigue = pulp.lpSum(data.loc[course, "coef fatigue"]*course_choice[course] for course in data.index) + W["alpha"]*ftg_max

  #Objective function
  problem += W["wc"] * credit_sum + W["wp"] * pop_sum - W["wh"] * hours_sum - W["wf"] * fatigue #satisfaction where w* are weights of the factors taken into account (sum of credits, popularity, time and overall fatigue)

  # Constraints
  #credits, hours, exams
  for req, value in req1.items():
    prefix, name = req.split("_")
    if prefix == "min":
      problem += pulp.lpSum(data.loc[course, name]*course_choice[course] for course in data.index) >= req1[req], f"min_{name}"
    else: #max
      problem += pulp.lpSum(data.loc[course, name]*course_choice[course] for course in data.index) <= req1[req], f"max_{name}"
  #categories m, f, n, j
  for req, value in req_cat.items():
    prefix, cat = req.split("_")
    if prefix == "min":
      problem += pulp.lpSum((data.loc[course, "categories"]== cat)*course_choice[course] for course in data.index) >= req_cat[req], f"min_{cat}"
    else: #max
      problem += pulp.lpSum((data.loc[course, "categories"]==cat)*course_choice[course] for course in data.index) <= req_cat[req], f"max_{cat}"
  #Solver
  solver = pulp.PULP_CBC_CMD()
  status_code = problem.solve(solver)
  status = pulp.LpStatus[status_code]

  #Results
  choice = {course : course_choice[course].varValue for course in course_choice} #1/0 as course choice from binary ILP
  final_choice = [course for course, val in choice.items() if val == 1] #list of chosen courses
  satisfaction = pulp.value(problem.objective)

  #Total credits, hours, exams
  exams = sum(data.loc[course, "exams"] for course in final_choice)
  credits = sum(data.loc[course, "credits"] for course in final_choice)
  hours = sum(data.loc[course, "hours"] for course in final_choice)
  totals = {
      "exams" : int(exams),
      "credits": int(credits),
      "hours": int(hours)
  }

  return {
      "status" : status,
      "choice" : choice,
      "final_choice" : final_choice,
      "totals" : totals,
      "satisfaction" : satisfaction
  }

weights = ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H']
for weight_option in weights:
  result = credit_solver(data, requirements, req1, req_cat, weight_option)
  print(f"Result for option {weight_option}:\n")
  print(f"Status: {result["status"]}\nCourse choice:")
  for course in result["final_choice"]:
    print(course, end=" ")
  #for course,val in result["choice"].items():
    #print(f"{course} : {val},", end=" ")
  print(f"\nTotals: {result["totals"]}\nSatisfaction: {result["satisfaction"]}\n\n")






Result for option A:

Status: Optimal
Course choice:
ANA3 VOAF PRC1 JAZ1 JAZ2 SSM1 SOS1 ZEL1 
Totals: {'exams': 3, 'credits': 30, 'hours': 29}
Satisfaction: -53.2


Result for option B:

Status: Optimal
Course choice:
ANA3 LIP VOAF JAZ1 JAZ2 SSM1 SOS1 ZEL1 TV1 
Totals: {'exams': 4, 'credits': 30, 'hours': 30}
Satisfaction: -40.35000000000001


Result for option C:

Status: Optimal
Course choice:
ANA3 VOAF PRC1 JAZ1 JAZ2 SSM1 SOS1 ZEL1 
Totals: {'exams': 3, 'credits': 30, 'hours': 29}
Satisfaction: -53.900000000000006


Result for option D:

Status: Optimal
Course choice:
ANA3 VOAF PRC1 JAZ1 JAZ2 SSM1 SOS1 ZEL1 
Totals: {'exams': 3, 'credits': 30, 'hours': 29}
Satisfaction: -61.99999999999999


Result for option E:

Status: Optimal
Course choice:
ANA3 NMA1 LIP VOAF PRC1 JAZ1 JAZ2 SSM1 SOS1 ZEL1 TV1 
Totals: {'exams': 5, 'credits': 38, 'hours': 38}
Satisfaction: 20.2


Result for option F:

Status: Optimal
Course choice:
NMA1 DIMA3 LIP VOAF PRC1 JAZ1 JAZ2 SSM1 SOS1 ZEL1 TV1 
Totals: {'ex

In [ ]:
def credit_solver_dual(data: pd.DataFrame, requirements: dict, req1 : dict, req_cat : dict, w_choice = 'A'):
  problem = pulp.LpProblem("Credit_Solver-dual", pulp.LpMinimize)
  #problem variables
  y = {i: pulp.LpVariable(f"y_{i}", lowBound=0) for i in range(1,12)}
  #new variables to replace constraint x_i <=1
  z = {i: pulp.LpVariable(f"z_{i}", lowBound=0) for i in range(1,14)} #corresponds to y_12,...,y_24 in pdf
  #weights of factors in the objective function
  weights = {
    "A" : {"wc": 0.1, "wp": 0.02, "wh": 1.0, "wf": 1.0, "alpha": 2},
    "B": {"wc": 0.05, "wp": 0.05, "wh": 0.8, "wf": 1.2, "alpha": 2},
    "C": {"wc": 0.5,  "wp": 0.02, "wh": 0.8, "wf": 1.5, "alpha": 3},
    "D": {"wc": 0.0,  "wp": 0.02, "wh": 1.2, "wf": 1.0, "alpha": 4},
    "E": {"wc": 1.0,  "wp": 0.02, "wh": 0.5, "wf": 0.2, "alpha": 2},
    "F": {"wc": 0.2,  "wp": 0.1,  "wh": 0.8, "wf": 0.8, "alpha": 2},
    "G": {"wc": 0.05, "wp": 0.02, "wh": 1.0, "wf": 0.5, "alpha": 3},
    "H": {"wc": 0.3,  "wp": 0.03, "wh": 0.9, "wf": 0.7, "alpha": 2.5}
             }
  W = weights[w_choice]
  #Objective function
  #in pdf: b_1 corresponds to b = (b^~_1 ... b^~_11) and b_2 corresponds to (b^~_12 ... b^~_24)
  #coefficients of objective function (vector b in min (b_1)^(T)*y + (b_2)^(T)*z)
  #in the right order
  b = {
      1: -req1["min_credits"],
      2: req1["max_credits"],
      3: -req1["min_exams"],
      4: req1["max_exams"],
      5: req1['max_hours'],
      6: -req_cat["min_m"],
      7: -req_cat["min_f"],
      8: -req_cat["min_n"],
      9: -req_cat["min_j"],
      10: req_cat["max_j"],
      11: 30     #ftg_max constraint
  }
  #Objective function
  problem += pulp.lpSum(b[i+1]*y[i+1] for i in range(len(b)))+pulp.lpSum(z[i] for i in z) #min (b_1)^(T)*y + (b_2)^(T)*z

  # Constraints
  #types of constraints
  #type 1
  t_1 = [1, 2, 3, 10]
  #type 2
  t_2 = [4,6,11,12]
  #type 3
  t_3 = [5,9]
  #type 4
  t_4 = [7,8]
  #type 5
  t_5 = [13]
  #type 6
  t_6 = [14]
  for i in range(1,14):
    hours_i = data["hours"].iloc[i-1]
    exams_i = data["exams"].iloc[i-1]
    credits_i = data["credits"].iloc[i-1]
    popularity_i = data["popularity"].iloc[i-1]
    coef_fatigue_i = data["coef fatigue"].iloc[i-1]
    beta_i = W["wc"]*credits_i + W["wp"] * popularity_i - W["wh"] * hours_i - W["wf"] * coef_fatigue_i
    if i in t_1: #type1-constraint:
      problem += (-credits_i*y[1] +credits_i*y[2] -exams_i*y[3] +exams_i*y[4]+hours_i*y[5]-1*y[6]+credits_i*y[11]+z[i]) >=  beta_i, f"constr_{i}"
    elif i in t_2: #type2-constraint:
      problem += (-credits_i*y[1] +credits_i*y[2] -exams_i*y[3] +exams_i*y[4]+hours_i*y[5]-1*y[8]+credits_i*y[11]+z[i]) >=  beta_i, f"constr_{i}"
    elif i in t_3: #type3-constraint
      problem += (-credits_i*y[1] +credits_i*y[2] -exams_i*y[3] +exams_i*y[4]+hours_i*y[5]-1*y[7]+credits_i*y[11]+z[i]) >=  beta_i, f"constr_{i}"
    elif i in t_4: #type4-constraint
      problem += (-credits_i*y[1] +credits_i*y[2] -exams_i*y[3] +exams_i*y[4]+hours_i*y[5]-1*y[9]+y[10]+credits_i*y[11]+z[i]) >=  beta_i, f"constr_{i}"
    elif i in t_5: #type5-constraint
      problem += (-credits_i*y[1] +credits_i*y[2] -exams_i*y[3] +exams_i*y[4]+hours_i*y[5]+credits_i*y[11]+z[i]) >=  beta_i, f"constr_{i}"
  #type6-constraint
  problem += (-1*y[11]) >= -W["alpha"] * W["wf"], f"constr_{14}"

  #Solver
  solver = pulp.PULP_CBC_CMD()
  status_code = problem.solve(solver)
  status = pulp.LpStatus[status_code]

  #Results
  y = {i : val.value() for i, val in y.items()}
  z = {i : val.value() for i, val in z.items()}
  satisfaction = pulp.value(problem.objective)

  #Total credits, hours, exams
  return {
    "status": status,
    "y": y,
    "z": z,
    "objective": satisfaction
}

weights = ['A', 'B', 'C', 'D', 'E', 'F', 'G', 'H']
for weight_option in weights:
  result = credit_solver_dual(data, requirements, req1, req_cat, weight_option)
  print(f"Result for option {weight_option}:\n")
  print(f"Status: {result["status"]}\n")
  print(f"\nY: {result["y"]}\nZ: {result["z"]}\nSatisfaction: {result["objective"]}\n\n\n")

Result for option A:

Status: Optimal


Y: {1: 1.66, 2: 0.0, 3: 0.28, 4: 0.0, 5: 0.0, 6: 0.38, 7: 1.86, 8: 0.0, 9: 1.58, 10: 0.0, 11: 0.0}
Z: {1: 0.0, 2: 0.0, 3: 0.0, 4: 0.0, 5: 0.0, 6: 0.44, 7: 0.4, 8: 0.0, 9: 0.0, 10: 0.0, 11: 1.62, 12: 0.76, 13: 0.16}
Satisfaction: -53.04000000000001



Result for option B:

Status: Optimal


Y: {1: 1.5777778, 2: 0.0, 3: 0.0, 4: 0.0, 5: 0.0, 6: 0.0, 7: 1.3833333, 8: 0.0, 9: 0.89444444, 10: 0.0, 11: 0.0}
Z: {1: 0.0, 2: 0.56111111, 3: 0.0, 4: 0.93333333, 5: 0.0, 6: 2.0111111, 7: 1.0, 8: 0.0, 9: 0.0, 10: 0.80555556, 11: 3.2055556, 12: 1.8833333, 13: 2.3277778}
Satisfaction: -37.77777838000001



Result for option C:

Status: Optimal


Y: {1: 1.6, 2: 0.0, 3: 0.2, 4: 0.0, 5: 0.0, 6: 0.8, 7: 2.7, 8: 0.0, 9: 2.5, 10: 0.0, 11: 0.0}
Z: {1: 0.0, 2: 0.0, 3: 0.0, 4: 0.0, 5: 0.0, 6: 0.6, 7: 0.4, 8: 0.0, 9: 0.0, 10: 0.0, 11: 2.2, 12: 0.8, 13: 0.4}
Satisfaction: -53.50000000000001



Result for option D:

Status: Optimal


Y: {1: 2.0, 2: 0.0, 3: 0.2, 4: 0.0, 5: 0.